# synthpose-webots — Geração de dataset COCO-pose do NAO no Kaggle

Este notebook roda o pipeline de geração de dataset sintético (Blender 5.1.2 headless + Cycles GPU)
diretamente em um **Kaggle Notebook**.

## Antes de rodar (menu à direita do Kaggle)
1. **Settings → Accelerator → GPU** (T4 x2 ou P100). Necessário para o Cycles renderizar rápido.
2. **Settings → Internet → On**. Necessário para baixar o Blender e clonar o repositório.

## O que o notebook faz
1. Baixa e extrai o **Blender 5.1.2** (Linux x64).
2. Clona o repositório `synthpose-webots`.
3. Instala `pyyaml` no Python embutido do Blender (numpy já vem embutido; `cv2`/`pycocotools`
   só são usados na validação, que roda no Python do Kaggle).
4. Habilita a **GPU no Cycles** (OptiX → CUDA) via um wrapper.
5. Roda `dataset_generator_phobos.py` gravando em `/kaggle/working/output`.
6. Valida o JSON COCO e mostra uma amostra renderizada.
7. Compacta a saída para download.


## 1. Configuração

Defina o **job** na célula abaixo: `TOTAL_SAMPLES` (total do dataset inteiro),
`WORLD_SIZE` (quantas máquinas Kaggle) e `RANK` (índice desta máquina, de 0).

Cada máquina pega uma **fatia disjunta** (`shard_range`) e, dentro dela, divide o
trabalho entre as **GPUs locais** — 1 processo Blender por T4 (a T4 x2 do Kaggle
gera em paralelo). Nomes de arquivo e sementes nunca colidem entre máquinas nem
entre GPUs, então o merge no fim só renumera os IDs COCO.

**2 máquinas:** mesmo `TOTAL_SAMPLES`/`WORLD_SIZE=2`, muda só `RANK` (0 e 1).
**1 máquina de 2 T4:** deixe `WORLD_SIZE=1, RANK=0` — as duas GPUs já são usadas.
O Kaggle limita a sessão a ~9h; comece com `TOTAL_SAMPLES` pequeno (ex.: 40) para
validar antes do lote grande.

> **Repositório privado?** Se `git clone` falhar por autenticação, faça upload do repo como um
> *Kaggle Dataset* e aponte `REPO_DIR` para `/kaggle/input/<seu-dataset>/synthpose-webots`
> (pule a célula de clone).

In [ ]:
import os

# --- Blender ---
BLENDER_VERSION = "5.1.2"
BLENDER_SERIES  = "Blender5.1"
BLENDER_URL = f"https://download.blender.org/release/{BLENDER_SERIES}/blender-{BLENDER_VERSION}-linux-x64.tar.xz"

# --- Repositorio ---
REPO_URL    = "https://github.com/Rinaldots/synthpose-webots.git"
REPO_BRANCH = "main"
REPO_DIR    = "/kaggle/working/synthpose-webots"   # ajuste p/ /kaggle/input/... se usar Dataset

# --- Job multi-maquina (orquestracao) ---
# Declare o job UMA vez e mude apenas RANK em cada maquina Kaggle:
#   Maquina 0 -> RANK=0 ;  Maquina 1 -> RANK=1 ;  ...
# O gerador deriva --start/--num de uma fatia DISJUNTA (ver nao_coco_pose.sharding),
# entao os nomes de arquivo e as sementes nunca colidem entre maquinas.
TOTAL_SAMPLES = 10000   # total do dataset inteiro, somando TODAS as maquinas
WORLD_SIZE    = 2       # quantas maquinas Kaggle vao rodar
RANK          = 0       # <<< MUDE POR MAQUINA: 0 na primeira, 1 na segunda, ...

OUT_DIR = "/kaggle/working/output"    # persiste e fica disponivel p/ download
USE_GPU = True

# --- Monitoramento remoto (opcional, via ntfy.sh) ---
# Deixe "" para desativar. Com um topico setado, cada maquina PUBLICA seu status
# (GPU + progresso) e voce agrega tudo no seu PC com:
#   python dataset_generator/scripts/monitor.py --subscribe <topico> --world-size 2
# Use um nome LONGO e ALEATORIO: funciona como senha (quem souber o topico, le).
NTFY_TOPIC = "synthpose-rinaldo-9f3a2b7c1d"   # ex.: "synthpose-rinaldo-9f3a2b7c1d"

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
assert 0 <= RANK < WORLD_SIZE, "RANK deve estar em [0, WORLD_SIZE)"
print(f"Config OK  |  job total={TOTAL_SAMPLES}  world_size={WORLD_SIZE}  ESTA MAQUINA rank={RANK}"
      + (f"  ntfy={NTFY_TOPIC}" if NTFY_TOPIC else "  (ntfy off)"))

## 2. Baixar e extrair o Blender 5.1.2

In [ ]:
import os, shutil, subprocess, glob, tarfile, urllib.request

TARBALL = f"{WORK}/blender.tar.xz"
BLENDER_HOME = f"{WORK}/blender-{BLENDER_VERSION}-linux-x64"

if not os.path.isdir(BLENDER_HOME):
    # download.blender.org recusa User-Agent nao-navegador (HTTP 403);
    # por isso enviamos um User-Agent de navegador.
    if not os.path.exists(TARBALL) or os.path.getsize(TARBALL) < 1_000_000:
        print("Baixando Blender...", BLENDER_URL)
        req = urllib.request.Request(BLENDER_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(TARBALL, "wb") as f:
            shutil.copyfileobj(r, f)
    print("Extraindo...")
    with tarfile.open(TARBALL) as t:
        t.extractall(WORK)

BLENDER = os.path.join(BLENDER_HOME, "blender")
assert os.path.exists(BLENDER), f"blender nao encontrado em {BLENDER}"

# Bibliotecas de sistema que o Blender headless costuma exigir no Kaggle
subprocess.run("apt-get -qq update && apt-get -qq install -y libxrender1 libxi6 libxxf86vm1 "
               "libxfixes3 libxkbcommon0 libsm6 libgl1 libegl1 2>/dev/null || true",
               shell=True)

print(subprocess.run([BLENDER, "--version"], capture_output=True, text=True).stdout)

## 3. Clonar o repositório
Pule esta célula e ajuste `REPO_DIR` na célula 1 se o repo estiver como *Kaggle Dataset*.

In [ ]:
import os, subprocess

# Clona se ainda nao existe; se JA existe, SINCRONIZA com o remoto (forca hard reset
# p/ pegar as ultimas alteracoes sem risco de conflito). Assim, re-rodar esta celula
# sempre traz a versao mais recente do branch.
if REPO_DIR.startswith("/kaggle/working"):
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH],
                       check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"],
                       check=True)
        print("Repo sincronizado com origin/" + REPO_BRANCH)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                       check=True)
        print("Repo clonado")

DG = os.path.join(REPO_DIR, "dataset_generator")
BLENDER_DIR = os.path.join(DG, "blender")
assert os.path.isfile(os.path.join(BLENDER_DIR, "nao_blender.blend")), \
    f"nao_blender.blend nao encontrado em {BLENDER_DIR}"
print("Repo pronto em:", REPO_DIR)

## 4. Instalar `pyyaml` no Python embutido do Blender
O gerador roda dentro do Blender, então a dependência precisa ir no Python **dele**
(não no do Kaggle). `numpy` já vem embutido.

In [ ]:
import glob, subprocess

BPY = glob.glob(os.path.join(BLENDER_HOME, "*", "python", "bin", "python3*"))
BPY = sorted(BPY)[0]
print("Python do Blender:", BPY)

subprocess.run([BPY, "-m", "ensurepip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "--upgrade", "pip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "pyyaml"], check=True)
print(subprocess.run([BPY, "-c", "import yaml, numpy; print('yaml', yaml.__version__, '| numpy', numpy.__version__)"],
                     capture_output=True, text=True).stdout)

## 5. Wrapper para habilitar a GPU no Cycles
O `dataset_generator_phobos.py` define o engine como CYCLES mas deixa o *device* em CPU.
Este wrapper é passado com um `--python` **antes** do gerador para ligar OptiX/CUDA.

In [ ]:
GPU_WRAPPER = os.path.join(BLENDER_DIR, "kaggle_enable_gpu.py")

with open(GPU_WRAPPER, "w") as f:
    f.write('''
import bpy

prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for dtype in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = dtype
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == dtype for d in prefs.devices):
        chosen = dtype
        break

enabled = 0
if chosen:
    for d in prefs.devices:
        d.use = (d.type == chosen)
        enabled += int(d.use)
    for sc in bpy.data.scenes:
        sc.cycles.device = "GPU"

print(f"[KAGGLE-GPU] compute_device_type={chosen} devices_enabled={enabled}")
''')

print("Wrapper escrito em", GPU_WRAPPER)

## 6. Gerar (1 processo por GPU) + monitor
Detecta as GPUs (T4 x2), pega a fatia desta máquina (`shard_range`) e a divide entre
as GPUs locais: **um Blender por GPU** (`CUDA_VISIBLE_DEVICES`), cada um gravando em
`output/gpu{i}`. `--resume` pula PNGs já feitos (retomável — re-rode a célula pra
continuar). Um **monitor** mostra a cada 15s as GPUs (`nvidia-smi`) + o progresso
somando todas as GPUs; com `NTFY_TOPIC` setado, publica p/ o painel no seu PC.

In [ ]:
import subprocess, os, sys, threading

sys.path.insert(0, os.path.join(DG, "scripts"))
sys.path.insert(0, os.path.join(DG, "src"))
from monitor import watch                       # count_done soma output/gpu*/images
from nao_coco_pose.sharding import shard_range

# Fatia desta MAQUINA (rank) dentro do job inteiro.
mshard = shard_range(TOTAL_SAMPLES, WORLD_SIZE, RANK)
M_START, M_NUM = mshard.start, mshard.num
print(f"Maquina rank {RANK}/{WORLD_SIZE}: start={M_START} num={M_NUM}")

n_gpu = max(1, subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("UUID"))
print("GPUs detectadas:", n_gpu)

# Divide a fatia da maquina entre as GPUs locais (contiguo, a partir de M_START).
base, rem = divmod(M_NUM, n_gpu)
shards, cursor = [], M_START
for g in range(n_gpu):
    num = base + (1 if g < rem else 0)
    shards.append({"gpu": str(g), "start": cursor, "num": num, "out": f"{OUT_DIR}/gpu{g}"})
    cursor += num

os.makedirs(OUT_DIR, exist_ok=True)
procs = []
for sh in shards:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=sh["gpu"])   # fixa 1 GPU por processo
    cmd = [BLENDER, "--background", "nao_blender.blend"]
    if USE_GPU:
        cmd += ["--python", "kaggle_enable_gpu.py"]
    # --resume: pula PNGs ja feitos (retomavel sem duplicar). Mantenha os mesmos
    # TOTAL_SAMPLES/WORLD_SIZE/RANK/n_gpu p/ retomar de onde parou.
    cmd += ["--python", "dataset_generator_phobos.py",
            "--", "--num", str(sh["num"]), "--start", str(sh["start"]),
            "--out", sh["out"], "--resume"]
    log = open(f"{OUT_DIR}/gpu{sh['gpu']}.log", "w")
    print(f"GPU {sh['gpu']}: start={sh['start']} num={sh['num']}  (log: {log.name})")
    procs.append((sh, subprocess.Popen(cmd, cwd=BLENDER_DIR, env=env,
                                       stdout=log, stderr=subprocess.STDOUT)))

# Monitor: 1 thread p/ a maquina toda — mostra as GPUs (nvidia-smi) + progresso
# somando gpu0+gpu1; publica no ntfy se NTFY_TOPIC estiver setado.
stop = threading.Event()
mon = threading.Thread(target=watch, kwargs=dict(
        out_dir=OUT_DIR, num=M_NUM, interval=15.0, stop=stop,
        rank=RANK, ntfy_topic=(NTFY_TOPIC or None)), daemon=True)
mon.start()
try:
    for sh, p in procs:
        print(f"GPU {sh['gpu']} terminou  rc={p.wait()}")
finally:
    stop.set(); mon.join(timeout=17)
    for sh in shards:                            # tail do log de cada GPU
        with open(f"{OUT_DIR}/gpu{sh['gpu']}.log") as f:
            tail = f.readlines()[-2:]
        print(f"--- gpu{sh['gpu']} (fim do log) ---\n{''.join(tail)}")

## 7. Validar o JSON COCO e visualizar uma amostra

In [ ]:
import glob, subprocess, json

# Valida o JSON COCO de cada GPU (roda no Python do Kaggle; pycocotools ja disponivel).
# Busca recursiva cobre output/gpu*/annotations/ (1 processo por GPU).
val = os.path.join(DG, "scripts", "validate_dataset.py")
ann = sorted(glob.glob(os.path.join(OUT_DIR, "**", "person_keypoints_*.json"), recursive=True))
env = dict(os.environ, PYTHONPATH=os.path.join(DG, "src"))
for a in ann:
    print("===", os.path.relpath(a, OUT_DIR), "===")
    subprocess.run(["python", val, a], env=env)

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Busca recursiva cobre output/gpu*/images/** (1 processo por GPU).
imgs = sorted(glob.glob(os.path.join(OUT_DIR, "**", "images", "**", "*.png"), recursive=True))
print(f"{len(imgs)} imagens geradas")
if imgs:
    n = min(4, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = [axes] if n == 1 else axes
    for ax, p in zip(axes, imgs[:n]):
        ax.imshow(mpimg.imread(p)); ax.set_title(os.path.basename(p)); ax.axis("off")
    plt.tight_layout(); plt.show()

## 8. Compactar a saída para download
O `.zip` aparece na aba **Output** / **Data** do Kaggle para download.

In [ ]:
import shutil
# Nomeia o zip por RANK para os downloads das varias maquinas nao colidirem no seu PC.
zip_base = f"/kaggle/working/nao_dataset_rank{RANK}of{WORLD_SIZE}"
zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
print("Pronto:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
print(f"\nBaixe este zip de CADA maquina, depois junte tudo localmente com:")
print(f"  python scripts/merge_datasets.py nao_dataset_rank*.zip --out output_merged")